# Практикум 14. Качество кластеризации

Сопровождает интерактивную лабораторию №14 («Оценка качества кластеризации»). Тема курса: силуэт, индексы Дэвиса–Болдина и Калински–Харабаса

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *

# Учебный корпус: 4 темы. noise - доля "посторонних" слов (чем больше, тем труднее).
df = make_corpus(n_per_class=150, noise=0.45, seed=0)
# Свой корпус вместо учебного: df = pd.read_csv("texts.csv")  # столбцы: text, label
print(df.label.value_counts().to_dict())
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score

X = TfidfVectorizer().fit_transform(df.text); y = df.label.values
Xr = make_pipeline(TruncatedSVD(30, random_state=0), Normalizer(copy=False)).fit_transform(X)

## 1. Три метрики для k от 2 до 10

In [ ]:
ks = range(2, 11); rows = []
for k in ks:
    lab = KMeans(k, n_init=10, random_state=0).fit_predict(Xr)
    rows.append((k, silhouette_score(Xr, lab), davies_bouldin_score(Xr, lab), calinski_harabasz_score(Xr, lab), adjusted_rand_score(y, lab)))
res = pd.DataFrame(rows, columns=["k", "silhouette", "davies_bouldin", "calinski_harabasz", "ARI"]).set_index("k")
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
for a, (c, t) in zip(ax, [("silhouette", "Силуэт (больше лучше)"), ("davies_bouldin", "Дэвис-Болдин (меньше лучше)"), ("calinski_harabasz", "Калински-Харабас (больше лучше)"), ("ARI", "ARI с настоящими темами")]):
    res[c].plot(marker="o", ax=a, title=t)
plt.tight_layout(); plt.show()
res.round(3)

## 2. График силуэта для выбранного k

In [ ]:
k = 4
lab = KMeans(k, n_init=10, random_state=0).fit_predict(Xr)
s = silhouette_samples(Xr, lab)
y0 = 0; plt.figure(figsize=(6, 5))
for c in range(k):
    v = np.sort(s[lab == c]); plt.barh(range(y0, y0 + len(v)), v, height=1.0, label=f"кластер {c}"); y0 += len(v) + 10
plt.axvline(s.mean(), color="red", ls="--"); plt.xlabel("силуэт"); plt.title(f"k={k}, средний силуэт {s.mean():.3f}"); plt.legend(); plt.show()

## Задания

1. Увеличьте `noise` до 0.7. Как меняются метрики и согласуются ли они между собой?
2. Посчитайте метрики для DBSCAN или агломеративной кластеризации. Какая метрика им «не подходит» и почему?
3. Сравните выбор k по внутренним метрикам и по ARI. Когда ARI недоступен?